# YOLOv11m + ECA+CBAM - Architecture Dependence Test

Attaches the identical residual-gated ECA+CBAM block used in RGDA-YOLOv8m to
YOLOv11m, whose backbone already contains a C2PSA partial self-attention stage,
and trains it across the same three seeds under the same two-phase schedule.


All runs use the deduplicated 926-image dataset and the seed-42 80/20 split.


Environment Detection

In [1]:
# Detects whether the notebook is running on Kaggle, Colab, or local Jupyter and sets ROOT, SAVE_DIR and OUTPUT_DIR accordingly.
import os, sys

ON_KAGGLE = os.path.exists("/kaggle/input")
ON_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
ON_JUPYTER = not ON_KAGGLE and not ON_COLAB

if ON_KAGGLE:
    print("Running on KAGGLE")
    ROOT = "/kaggle/working"
elif ON_COLAB:
    print("Running on GOOGLE COLAB")
    ROOT = "/content"
else:
    print("Running on LOCAL JUPYTER")
    ROOT = "."

OUTPUT_DIR = os.path.join(ROOT, "attention_results")
DATA_DIR = os.path.join(ROOT, "data")
SAVE_DIR = os.path.join(ROOT, "saved_models")

for d in [OUTPUT_DIR, DATA_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"   Output  -> {OUTPUT_DIR}")
print(f"   Data    -> {DATA_DIR}")
print(f"   Models  -> {SAVE_DIR}")

Running on LOCAL JUPYTER
   Output  -> ./attention_results
   Data    -> ./data
   Models  -> ./saved_models


Verify Environment

In [2]:
# Prints package versions and confirms GPU availability and device name.
import torch, numpy as np, pandas as pd, cv2

print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"OpenCV  : {cv2.__version__}")
print(f"PyTorch : {torch.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")
    print(
        f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
else:
    print("   WARNING: No GPU -- training will be slow.")

try:
    import ultralytics, seaborn, tqdm, kagglehub, yaml

    print("ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!")
except ImportError as e:
    print(f"FAIL: Missing package: {e}")
    print("   Run: pip install ultralytics seaborn tqdm kagglehub pyyaml")

NumPy   : 2.2.6
Pandas  : 2.3.3
OpenCV  : 4.13.0
PyTorch : 2.10.0+cu128
Device  : cuda
   GPU    : NVIDIA GeForce RTX 3090
   VRAM   : 25.4 GB
ultralytics, seaborn, tqdm, kagglehub, pyyaml -- all good!


Config & Hyperparameters

In [3]:
# Defines the training and evaluation config: two-phase epochs and learning rates, batch size, image size, seeds, the confidence sweep grid, and the default evaluation thresholds.
EPOCHS_FROZEN = 10  # Phase 1: train only attention + head, backbone frozen
EPOCHS_FULL = 40  # Phase 2: unfreeze everything, fine-tune end-to-end
BATCH = 16
IMG_SIZE = 640
LR_FROZEN = 1e-3  # Higher LR for Phase 1 (only new layers training)
LR_FULL = 2e-4  # Lower LR for Phase 2 (avoid overwriting pretrained weights)
WEIGHT_DECAY = 5e-4

# Attention config
# ECA: kernel size is adaptive (uses log2 of channels) -- no hyperparams needed
# CBAM: reduction ratio for channel attention MLP
CBAM_REDUCTION = 16
CBAM_KERNEL = 7  # spatial attention conv kernel

# Evaluation config
# We evaluate across a sweep and also report at the optimal threshold.
CONF_SWEEP = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
]
CONF_DEFAULT = 0.25  # Standard YOLO default for fair comparison
IOU_THRESH = 0.5  # IoU threshold for TP/FP
MAX_IMAGES = None  # None = use full test set

print("Config loaded")
print(f"   Phase 1 : {EPOCHS_FROZEN} epochs, freeze backbone, LR={LR_FROZEN}")
print(f"   Phase 2 : {EPOCHS_FULL} epochs, full fine-tune, LR={LR_FULL}")
print(
    f"   Eval conf: sweep {CONF_SWEEP[0]}-{CONF_SWEEP[-1]}, default at {CONF_DEFAULT}"
)

Config loaded
   Phase 1 : 10 epochs, freeze backbone, LR=0.001
   Phase 2 : 40 epochs, full fine-tune, LR=0.0002
   Eval conf: sweep 0.1-0.7, default at 0.25


Dataset Loading

In [4]:
# Downloads the three Kaggle datasets, loads every annotation through the unified loader, and deduplicates before any split.
import kagglehub
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil

print("Downloading datasets via kagglehub ...")
path_1 = kagglehub.dataset_download("chitholian/annotated-potholes-dataset")
path_2 = kagglehub.dataset_download("andrewmvd/pothole-detection")
path_3 = kagglehub.dataset_download("ashishkumarak/training-setzip")
DATASET_ROOTS = {"chitholian": path_1, "andrewmvd": path_2, "ashishkumar": path_3}
print("Datasets ready")


def load_annotated_potholes(root, max_imgs=None):
    root = Path(root)
    xml_index = {p.stem: p for p in root.rglob("*.xml")}

    records = []
    for img_path in (list(root.rglob("*.jpg")) + list(root.rglob("*.png")))[:max_imgs]:
        gt_boxes = []
        xml_path = xml_index.get(img_path.stem)
        if xml_path is not None and xml_path.exists():
            try:
                tree = ET.parse(xml_path)
                for obj in tree.findall("object"):
                    bb = obj.find("bndbox")
                    gt_boxes.append(
                        [
                            float(bb.find("xmin").text),
                            float(bb.find("ymin").text),
                            float(bb.find("xmax").text),
                            float(bb.find("ymax").text),
                        ]
                    )
            except Exception:
                pass
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


def load_ashishkumar_csv(root, max_imgs=None):
    import pandas as pd

    root = Path(root)
    csv_path = root / "train" / "labels.csv"
    img_dir = root / "train" / "images"
    df = pd.read_csv(csv_path)
    grouped = df.groupby("ImageID")

    records = []
    img_paths = sorted(img_dir.glob("*.jpg"))[:max_imgs]
    for img_path in img_paths:
        gt_boxes = []
        if img_path.name in grouped.groups:
            for _, row in grouped.get_group(img_path.name).iterrows():
                gt_boxes.append(
                    [
                        float(row["XMin"]),
                        float(row["YMin"]),
                        float(row["XMax"]),
                        float(row["YMax"]),
                    ]
                )
        records.append({"image_path": img_path, "gt_boxes": gt_boxes})
    return records


all_records = []
for name, root in DATASET_ROOTS.items():
    if name == "ashishkumar":
        recs = load_ashishkumar_csv(root, MAX_IMAGES)
    else:
        recs = load_annotated_potholes(root, MAX_IMAGES)
    print(
        f"   {name}: {len(recs)} images ({sum(len(r['gt_boxes']) for r in recs)} gt boxes)"
    )
    all_records.extend(recs)

records = [r for r in all_records if r["gt_boxes"]]  # only annotated
print(f"\nTotal annotated before dedup: {len(records)} images")

import numpy as np
from PIL import Image

NORM_SIZE = (64, 64)
DEDUP_THRESHOLD = 1.0  # mean abs pixel diff (0-255 scale); true duplicates
# measured at 0.10-0.50, unrelated images much higher


def normalized_pixels(path):
    with Image.open(path) as img:
        return np.asarray(
            img.convert("L").resize(NORM_SIZE, Image.LANCZOS), dtype=np.float32
        ).ravel()


print("Computing normalized pixel arrays for dedup ...")
all_arrs = np.stack([normalized_pixels(r["image_path"]) for r in records])

keep_mask = np.ones(len(records), dtype=bool)
seen_arrs = []  # arrays of images already kept
for i in range(len(records)):
    if not keep_mask[i]:
        continue
    if seen_arrs:
        diffs = np.abs(np.stack(seen_arrs) - all_arrs[i]).mean(axis=1)
        if diffs.min() < DEDUP_THRESHOLD:
            keep_mask[i] = False
            continue
    seen_arrs.append(all_arrs[i])

# Diagnostic: show the distribution of nearest-neighbor diffs among the
# images that got removed, so the threshold can be sanity-checked directly
# rather than guessed at again if the final count still looks off.
removed_diffs = []
_seen_for_diag = []
for i in range(len(records)):
    if _seen_for_diag:
        d = np.abs(np.stack(_seen_for_diag) - all_arrs[i]).mean(axis=1).min()
        if not keep_mask[i]:
            removed_diffs.append(d)
    if keep_mask[i]:
        _seen_for_diag.append(all_arrs[i])
if removed_diffs:
    removed_diffs = np.array(removed_diffs)
    print(
        f"\nRemoved-pair diff stats: min={removed_diffs.min():.3f} "
        f"median={np.median(removed_diffs):.3f} max={removed_diffs.max():.3f}"
    )
    print(
        f"   (all removed pairs should sit well below DEDUP_THRESHOLD={DEDUP_THRESHOLD} "
        f"-- if max is close to the threshold, some may be false positives)"
    )

n_before = len(records)
records = [r for r, keep in zip(records, keep_mask) if keep]
n_after = len(records)
print(f"Total annotated after dedup: {n_after} images")
print(f"   Duplicates removed: {n_before - n_after}")


Datasets ready
   chitholian: 665 images (1740 gt boxes)
   andrewmvd: 665 images (1740 gt boxes)
   ashishkumar: 674 images (1371 gt boxes)

Total annotated before dedup: 2004 images
Computing normalized pixel arrays for dedup ...

Removed-pair diff stats: min=0.000 median=0.140 max=0.997
   (all removed pairs should sit well below DEDUP_THRESHOLD=1.0 -- if max is close to the threshold, some may be false positives)
Total annotated after dedup: 926 images
   Duplicates removed: 1078


Build YOLO Dataset (train/val split)

In [5]:
# Shuffles the deduplicated records under seed 42, splits 80/20, and writes the YOLO-format image and label directories plus data.yaml.
import random, yaml

random.seed(42)
random.shuffle(records)
split_idx = int(len(records) * 0.8)
train_recs, val_recs = records[:split_idx], records[split_idx:]
print(f"Train: {len(train_recs)} | Val: {len(val_recs)}")

YOLO_DIR = os.path.join(ROOT, "yolo_dataset")
DATA_YAML = f"{YOLO_DIR}/data.yaml"

for split in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(f"{YOLO_DIR}/{split}", exist_ok=True)


def convert_to_yolo(rec_list, split):
    """Write YOLO-format label files and copy images."""
    written = 0
    for rec in rec_list:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            continue
        h, w = img.shape[:2]
        dst_img = f"{YOLO_DIR}/images/{split}/{rec['image_path'].name}"
        shutil.copy(str(rec["image_path"]), dst_img)
        lbl_path = f"{YOLO_DIR}/labels/{split}/{rec['image_path'].stem}.txt"
        with open(lbl_path, "w") as f:
            for box in rec["gt_boxes"]:
                x1, y1, x2, y2 = box
                cx = ((x1 + x2) / 2) / w
                cy = ((y1 + y2) / 2) / h
                bw = (x2 - x1) / w
                bh = (y2 - y1) / h
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
        written += 1
    return written


n_train = convert_to_yolo(train_recs, "train")
n_val = convert_to_yolo(val_recs, "val")

data_cfg = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["pothole"],
}
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f)

print(f"YOLO dataset ready -- train:{n_train}, val:{n_val}")
print(f"   YAML: {DATA_YAML}")

Train: 740 | Val: 186
YOLO dataset ready -- train:740, val:186
   YAML: ./yolo_dataset/data.yaml


ECA Module Definition

**Efficient Channel Attention (ECA)** -- Wang et al., CVPR 2020.  
Replaces the MLP bottleneck in SE-Net with a single 1D conv over channels. Kernel size is determined adaptively from channel count: `k = ceil(log2(C)/2)*2 + 1`. This gives ~0 parameter overhead while capturing local cross-channel dependencies.

In [6]:
# Defines the ECA module, a single 1D convolution over channels with an adaptively sized kernel, wrapped in a zero-initialized residual gate.
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class ECA(nn.Module):
    """
    Efficient Channel Attention (ECA-Net).
    Wang et al., CVPR 2020 -- https://arxiv.org/abs/1910.03151

    Uses adaptive 1-D convolution over channels (no FC layers).
    Kernel size k is determined by the number of channels C:
        k = ceil(log2(C) / 2) * 2 + 1  (always odd)
    """

    def __init__(self, in_channels, gamma=2, b=1):
        super().__init__()
        # adaptive kernel size
        t = int(abs(math.log2(in_channels) / gamma) + b / gamma)
        k = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=(k - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)  # (B, C, 1, 1)
        y = y.squeeze(-1).transpose(-1, -2)  # (B, 1, C)
        y = self.conv(y)  # (B, 1, C)
        y = self.sigmoid(y)
        y = y.transpose(-1, -2).unsqueeze(-1)  # (B, C, 1, 1)
        return x * y.expand_as(x)


# Sanity check
dummy = torch.randn(2, 256, 20, 20)
eca = ECA(256)
out = eca(dummy)
n_params = sum(p.numel() for p in eca.parameters())
print(f"ECA module defined")
print(f"   Input  : {tuple(dummy.shape)}")
print(f"   Output : {tuple(out.shape)}")
print(f"   Params : {n_params}  (intentionally tiny)")

ECA module defined
   Input  : (2, 256, 20, 20)
   Output : (2, 256, 20, 20)
   Params : 5  (intentionally tiny)


CBAM Module Definition

**Convolutional Block Attention Module (CBAM)** -- Woo et al., ECCV 2018.  
Applies **channel attention** (what features to amplify) then **spatial attention** (where to focus). Addresses false positives by suppressing background texture.

In [7]:
# Defines the CBAM channel and spatial attention sub-modules.
class ChannelAttention(nn.Module):
    """CBAM channel sub-module -- shared MLP on avg+max pooled descriptors."""

    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(1, in_channels // reduction)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, in_channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    """CBAM spatial sub-module -- conv on channel-wise avg+max."""

    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return scale


class CBAM(nn.Module):
    """
    Convolutional Block Attention Module.
    Woo et al., ECCV 2018 -- https://arxiv.org/abs/1807.06521

    Applied AFTER ECA on the YOLOv11 neck feature maps.
    Channel attention -> spatial attention (sequential, as per paper).
    """

    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(in_channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.channel_att(x)  # channel-wise scaling
        x = x * self.spatial_att(x)  # spatial-wise scaling
        return x


# Sanity check
dummy = torch.randn(2, 256, 20, 20)
cbam = CBAM(256, CBAM_REDUCTION, CBAM_KERNEL)
out = cbam(dummy)
n_params = sum(p.numel() for p in cbam.parameters())
print(f"CBAM module defined")
print(f"   Input  : {tuple(dummy.shape)}")
print(f"   Output : {tuple(out.shape)}")
print(f"   Params : {n_params:,}")

CBAM module defined
   Input  : (2, 256, 20, 20)
   Output : (2, 256, 20, 20)
   Params : 8,290


ECA + CBAM Sequential Module

In [8]:
# Defines the sequential ECA_CBAM block, ECA channel recalibration followed by CBAM, behind a learnable scale initialised at zero so the block starts as an identity.
class ECA_CBAM(nn.Module):
    """
    Sequential ECA -> CBAM attention block.

    ECA first performs lightweight channel recalibration with almost no
    parameters. CBAM then refines both channel and spatial attention.
    Applied as a residual: output = x + attention(x).

    Residual connection prevents attention from completely suppressing
    features during early training when weights are random.
    """

    def __init__(
        self, in_channels, cbam_reduction=16, cbam_kernel=7, use_residual=True
    ):
        super().__init__()
        self.eca = ECA(in_channels)
        self.cbam = CBAM(in_channels, cbam_reduction, cbam_kernel)
        self.use_residual = use_residual
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        att = self.cbam(self.eca(x))
        if self.use_residual:
            # tanh(scale)  in  (-1, 1); starts near 0 at init
            return x + torch.tanh(self.scale) * (att - x)
        return att


# Verify
dummy = torch.randn(2, 256, 20, 20)
eca_cbam = ECA_CBAM(256)
out = eca_cbam(dummy)
n_params = sum(p.numel() for p in eca_cbam.parameters())
print(f"ECA_CBAM (residual) defined")
print(f"   Params : {n_params:,}")
print(f"   Scale  : {eca_cbam.scale.item():.4f}  (~0 at init = safe no-op)")
# Confirm output ~ input at init (scale~0 -> tanh(0)=0 -> residual passthrough)
max_diff = (out - dummy).abs().max().item()
print(f"   |out - input| max at init: {max_diff:.6f}  (should be ~ 0.0)")

ECA_CBAM (residual) defined
   Params : 8,296
   Scale  : 0.0000  (~0 at init = safe no-op)
   |out - input| max at init: 0.000000  (should be ~ 0.0)


Hook Injector: Insert ECA+CBAM into YOLOv11 Neck

We use **forward hooks** to inject attention after each C2f layer in the YOLOv11 neck. This avoids modifying the Ultralytics model source.



In [9]:
# Defines get_neck_modules and AttentionHookInjector, which locate the YOLOv11 neck layers by class name (C3k2, not C2f) and attach the attention block as forward hooks.
def get_neck_modules(yolo_detection_model):
    """
    Return neck feature-extraction modules from a YOLOv11 DetectionModel.
    YOLOv11 uses C3k2 (not C2f) in its neck. We match by class name so
    this works across YOLOv8/10/11 without importing version-specific classes.
    Returns the last 3 matching layers (FPN neck portion).
    """
    import torch.nn as nn

    # Resolve the Sequential of layers from DetectionModel
    if hasattr(yolo_detection_model, "model") and isinstance(
        yolo_detection_model.model, nn.Sequential
    ):
        layer_seq = yolo_detection_model.model
    else:
        children = list(yolo_detection_model.children())
        layer_seq = next((c for c in children if isinstance(c, nn.Sequential)), None)
        if layer_seq is None:
            raise RuntimeError(
                "Cannot locate the layer Sequential inside DetectionModel"
            )

    all_layers = list(layer_seq)
    print(f"   Total layers in model: {len(all_layers)}")

    # Match by class name -- covers C3k2 (v11), C2f (v8/v10), C2fAttn, etc.
    NECK_CLASS_NAMES = {"C3k2", "C2f", "C2fAttn", "RepC3", "C3"}
    candidates = []
    for idx, layer in enumerate(all_layers):
        cname = type(layer).__name__
        if cname in NECK_CLASS_NAMES:
            candidates.append((idx, layer, cname))

    print(f"   Found {len(candidates)} C3k2/C2f-type layers:")
    for idx, layer, cname in candidates:
        print(f"     layer[{idx:2d}]  {cname}")

    if not candidates:
        present = {type(l).__name__ for l in all_layers}
        raise RuntimeError(f"No neck-type layers found. Classes present: {present}")

    # Last 3 = neck portion (earlier ones are backbone)
    neck = candidates[-3:] if len(candidates) >= 3 else candidates
    return [layer for (_, layer, _) in neck]


class AttentionHookInjector:
    """
    Injects ECA_CBAM into YOLOv11 neck via forward hooks.

    Usage:
        injector = AttentionHookInjector(model.model)  # DetectionModel
        injector.attach()
        # ... train ...
        injector.detach()
    """

    def __init__(self, yolo_detection_model, cbam_reduction=16, cbam_kernel=7):
        import torch.nn as nn

        self._hooks = []
        self.attention_modules = nn.ModuleList()
        self._device = next(yolo_detection_model.parameters()).device

        neck_layers = get_neck_modules(yolo_detection_model)
        print(f"   Hooking {len(neck_layers)} neck layers")

        # Probe pass: discover real output channel counts
        real_channels = [None] * len(neck_layers)
        probe_hooks = []

        def make_probe(i):
            def hook(module, inp, out):
                t = out[0] if isinstance(out, (list, tuple)) else out
                real_channels[i] = t.shape[1]

            return hook

        for i, layer in enumerate(neck_layers):
            probe_hooks.append(layer.register_forward_hook(make_probe(i)))

        try:
            import torch

            dummy = torch.zeros(1, 3, 640, 640, device=self._device)
            with torch.no_grad():
                yolo_detection_model(dummy)
        except Exception as e:
            print(f"   Probe pass note: {e}")
        finally:
            for h in probe_hooks:
                h.remove()

        # Build attention modules at discovered channel widths
        for i, (layer, c) in enumerate(zip(neck_layers, real_channels)):
            if c is None:
                print(f"   Layer {i}: channel probe failed -- skipping")
                continue
            att = ECA_CBAM(c, cbam_reduction, cbam_kernel, use_residual=True).to(
                self._device
            )
            self.attention_modules.append(att)
            print(f"   Neck layer {i}: channels={c}  ECA_CBAM attached")

        self._neck_layers = neck_layers

    def attach(self):
        "Register forward hooks -- call before training."
        self._hooks = []
        for layer, att in zip(self._neck_layers, self.attention_modules):

            def make_hook(a):
                def hook(module, inp, out):
                    # Handle tuple outputs (some Ultralytics modules)
                    if isinstance(out, (list, tuple)):
                        return type(out)([a(out[0])] + list(out[1:]))
                    return a(out)

                return hook

            self._hooks.append(layer.register_forward_hook(make_hook(att)))
        print(f"  {len(self._hooks)} attention hooks attached")

    def detach(self):
        "Remove all hooks."
        for h in self._hooks:
            h.remove()
        self._hooks = []
        print("  Hooks removed")

    @property
    def parameters(self):
        return self.attention_modules.parameters()

    def n_params(self):
        return sum(p.numel() for p in self.attention_modules.parameters())


print("  get_neck_modules + AttentionHookInjector defined")


  get_neck_modules + AttentionHookInjector defined


Load YOLOv11m & Attach Attention

**Two-phase training strategy:**
- **Phase 1:** Backbone frozen. Only attention modules + detection head train. Higher LR. Lets attention weights stabilise before full fine-tuning.
- **Phase 2:** Full model unfrozen. Lower LR for smooth convergence.

In [10]:
# Clears any previous model from GPU memory and loads YOLOv11m with the attention injector attached.
import gc
from ultralytics import YOLO

# Clear old models from GPU memory
for _var in ["model", "injector"]:
    if _var in globals():
        del globals()[_var]
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("Loading YOLOv11m ...")
model = YOLO("yolo11m.pt")
model.to(DEVICE)

# Diagnostic: show what block types are actually inside this model
print("\nLayer class names (unique):")
from collections import Counter

layer_seq = model.model.model  # DetectionModel -> Sequential
names = [type(l).__name__ for l in list(layer_seq)]
for name, count in Counter(names).most_common():
    print(f"   {name}: {count}x")

live_injector = {"obj": None}


def _attach_on_train_start(trainer):
    inj = AttentionHookInjector(
        trainer.model, cbam_reduction=CBAM_REDUCTION, cbam_kernel=CBAM_KERNEL
    )
    inj.attach()
    live_injector["obj"] = inj

    if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
        last_group = (
            trainer.optimizer.param_groups[-1] if trainer.optimizer.param_groups else {}
        )
        new_group = {"params": list(inj.attention_modules.parameters())}
        new_group["lr"] = last_group.get("lr", LR_FROZEN)
        new_group["initial_lr"] = last_group.get("initial_lr", new_group["lr"])
        trainer.optimizer.add_param_group(new_group)

        sched = getattr(trainer, "scheduler", None)
        if sched is not None:
            n_groups = len(trainer.optimizer.param_groups)
            if hasattr(sched, "base_lrs") and len(sched.base_lrs) < n_groups:
                sched.base_lrs.append(sched.base_lrs[-1])
            if hasattr(sched, "lr_lambdas") and len(sched.lr_lambdas) < n_groups:
                sched.lr_lambdas.append(sched.lr_lambdas[-1])

        total = sum(p.numel() for p in trainer.model.parameters())
        print(
            f"  [callback] hooks attached + {inj.n_params():,} attention params "
            f"registered with optimizer  ({inj.n_params() / total * 100:.2f}% overhead)"
        )
    else:
        print(
            "  [callback] WARNING: trainer.optimizer not found -- "
            "attention params will NOT be trained this phase"
        )


def _detach_on_train_end(trainer):
    inj = live_injector.get("obj")
    if inj is not None:
        inj.detach()


model.add_callback("on_train_start", _attach_on_train_start)
model.add_callback("on_train_end", _detach_on_train_end)

print(
    "\n[FIXED] Hooks will attach via on_train_start callback during .train() below, "
    "not here -- pre-training attachment was the bug."
)

Loading YOLOv11m ...

Layer class names (unique):
   C3k2: 8x
   Conv: 7x
   Concat: 4x
   Upsample: 2x
   SPPF: 1x
   C2PSA: 1x
   Detect: 1x

[FIXED] Hooks will attach via on_train_start callback during .train() below, not here -- pre-training attachment was the bug.


Pre-flight checks before Phase 1

Cheap, fast checks (seconds, no real training) that
confirming gradients actually reach the attention parameters BEFORE committing to a multi-hour training run.

In [11]:
# Pre-flight checks: confirms the data and environment are sound and that gradients actually reach the attention parameters before any real training starts.
import os, torch

print("=" * 60)
print("PRE-FLIGHT CHECKS")
print("=" * 60)

ok = True

# Device
print(f"[1] Device: {DEVICE}", "(cuda)" if DEVICE == "cuda" else "(!) no GPU detected")
if DEVICE == "cuda":
    print(f"    GPU: {torch.cuda.get_device_name(0)}")

# Dataset file exists and is non-empty
print(f"[2] DATA_YAML: {DATA_YAML}")
if not os.path.exists(DATA_YAML):
    print("    FAIL: file does not exist")
    ok = False
else:
    import yaml

    with open(DATA_YAML) as f:
        d = yaml.safe_load(f)
    n_train = (
        len(os.listdir(os.path.join(d["path"], "images/train")))
        if os.path.exists(os.path.join(d["path"], "images/train"))
        else 0
    )
    n_val = (
        len(os.listdir(os.path.join(d["path"], "images/val")))
        if os.path.exists(os.path.join(d["path"], "images/val"))
        else 0
    )
    print(f"    train images: {n_train}   val images: {n_val}")
    if n_train == 0 or n_val == 0:
        print("    FAIL: train or val split is empty")
        ok = False

# Required config constants exist
for name in [
    "EPOCHS_FROZEN",
    "EPOCHS_FULL",
    "BATCH",
    "IMG_SIZE",
    "LR_FROZEN",
    "LR_FULL",
    "WEIGHT_DECAY",
    "CBAM_REDUCTION",
    "CBAM_KERNEL",
    "SAVE_DIR",
    "OUTPUT_DIR",
]:
    present = name in dir()
    print(f"[3] {name}: {'OK' if present else 'MISSING'} ({globals().get(name)})")
    if not present:
        ok = False

# Callback functions exist (defined in the cell above)
for name in ["_attach_on_train_start", "_detach_on_train_end"]:
    present = name in dir()
    print(f"[4] {name}: {'defined' if present else 'MISSING'}")
    if not present:
        ok = False

# THE IMPORTANT ONE - gradient flow check.
# Build a throwaway YOLOv11m + attention injector, run ONE forward + backward
# pass on a dummy batch, and confirm gradients actually reach the attention
# module's parameters. This is the exact failure we hit before (scale stuck
# at exactly 0.000000) but catchable in ~10 seconds instead of a full epoch.
print("[5] Gradient flow check (dummy forward/backward, no real training) ...")
from ultralytics import YOLO

_test_model = YOLO("yolo11m.pt")
_test_model.to(DEVICE)
_test_injector = AttentionHookInjector(
    _test_model.model, cbam_reduction=CBAM_REDUCTION, cbam_kernel=CBAM_KERNEL
)
_test_injector.attach()

_test_model.model.train()
_dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE, requires_grad=False)
_out = _test_model.model(_dummy_input)


# Collect every tensor out of whatever structure comes back (tensor, list/
# tuple, or dict varies by Ultralytics version / head), then sum into a
# scalar loss just to trigger backward().
def _collect_tensors(x):
    if torch.is_tensor(x):
        return [x]
    if isinstance(x, dict):
        out = []
        for v in x.values():
            out.extend(_collect_tensors(v))
        return out
    if isinstance(x, (list, tuple)):
        out = []
        for v in x:
            out.extend(_collect_tensors(v))
        return out
    return []


_tensors = _collect_tensors(_out)
assert _tensors, (
    f"No tensors found in model output (type={type(_out)}) -- cannot run grad check"
)
_loss = sum(t.float().sum() for t in _tensors)
_loss.backward()

grad_ok = True
for name, p in _test_injector.attention_modules.named_parameters():
    if "scale" in name:
        has_grad = p.grad is not None and p.grad.abs().item() > 0
        print(
            f"    {name}: grad={'present, nonzero' if has_grad else 'MISSING or zero'}"
        )
        if not has_grad:
            grad_ok = False
if not grad_ok:
    print("    FAIL: attention parameters are not receiving gradients")
    ok = False
else:
    print("    OK: gradients reach the attention parameters")

_test_injector.detach()
del _test_model, _test_injector, _out, _loss
import gc

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("=" * 60)
print(
    "ALL CHECKS PASSED -- safe to run Phase 1"
    if ok
    else "CHECKS FAILED -- do not start training, see failures above"
)
print("=" * 60)
assert ok, "Pre-flight checks failed -- fix the issues above before training"

PRE-FLIGHT CHECKS
[1] Device: cuda (cuda)
    GPU: NVIDIA GeForce RTX 3090
[2] DATA_YAML: ./yolo_dataset/data.yaml
    train images: 740   val images: 186
[3] EPOCHS_FROZEN: OK (10)
[3] EPOCHS_FULL: OK (40)
[3] BATCH: OK (16)
[3] IMG_SIZE: OK (640)
[3] LR_FROZEN: OK (0.001)
[3] LR_FULL: OK (0.0002)
[3] WEIGHT_DECAY: OK (0.0005)
[3] CBAM_REDUCTION: OK (16)
[3] CBAM_KERNEL: OK (7)
[3] SAVE_DIR: OK (./saved_models)
[3] OUTPUT_DIR: OK (./attention_results)
[4] _attach_on_train_start: defined
[4] _detach_on_train_end: defined
[5] Gradient flow check (dummy forward/backward, no real training) ...
   Total layers in model: 24
   Found 8 C3k2/C2f-type layers:
     layer[ 2]  C3k2
     layer[ 4]  C3k2
     layer[ 6]  C3k2
     layer[ 8]  C3k2
     layer[13]  C3k2
     layer[16]  C3k2
     layer[19]  C3k2
     layer[22]  C3k2
   Hooking 3 neck layers
   Neck layer 0: channels=256  ECA_CBAM attached
   Neck layer 1: channels=512  ECA_CBAM attached
   Neck layer 2: channels=512  ECA_CBAM attached


Metric Helpers

In [12]:
# Defines compute_iou, the AP computation and evaluate_model, the shared scoring functions used for every evaluation in this notebook.
def compute_iou(boxA, boxB):
    """IoU between two [x1,y1,x2,y2] boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    aA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    aB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / (aA + aB - inter + 1e-9)


def compute_ap_voc11(recalls, precisions):
    """Legacy PASCAL VOC-2007 11-point AP. Kept for continuity only."""
    ap = 0.0
    for thr in np.linspace(0, 1, 11):
        p_at_r = [p for r, p in zip(recalls, precisions) if r >= thr]
        ap += max(p_at_r) if p_at_r else 0.0
    return ap / 11.0


def compute_ap(recalls, precisions):
    """COCO-style 101-point interpolated AP."""
    if len(recalls) == 0:
        return 0.0
    mrec = np.concatenate(([0.0], np.asarray(recalls, dtype=float), [1.0]))
    mpre = np.concatenate(([1.0], np.asarray(precisions, dtype=float), [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    _trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(_trapz(np.interp(x, mrec, mpre), x))


def evaluate_model(yolo_model, recs, conf_thresh=0.25, iou_thresh=0.5):
    """
    Run inference on recs and return metrics.

    Returns dict with mAP@0.5, Precision, Recall, F1, TP, FP, FN, FPS.
    """
    all_preds, all_gt = [], []
    fps_times = []

    for rec in recs:
        img = cv2.imread(str(rec["image_path"]))
        if img is None:
            continue
        t0 = time.perf_counter()
        res = yolo_model.predict(img, conf=0.001, verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        fps_times.append(time.perf_counter() - t0)

        preds = []
        if res and res[0].boxes is not None and len(res[0].boxes.xyxy) > 0:
            for box, score in zip(
                res[0].boxes.xyxy.cpu().numpy(), res[0].boxes.conf.cpu().numpy()
            ):
                preds.append({"box": box.tolist(), "score": float(score)})
        all_preds.append(preds)
        all_gt.append(rec["gt_boxes"])

    # Compute TP/FP/FN at the requested deployment threshold
    ttp = tfp = tfn = 0
    for preds, gts in zip(all_preds, all_gt):
        preds_at_thresh = [p for p in preds if p["score"] >= conf_thresh]
        matched_gt = set()
        for pred in sorted(preds_at_thresh, key=lambda x: -x["score"]):
            best_iou, best_j = 0.0, -1
            for j, gt in enumerate(gts):
                if j in matched_gt:
                    continue
                iou = compute_iou(pred["box"], gt)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_thresh:
                ttp += 1
                matched_gt.add(best_j)
            else:
                tfp += 1
        tfn += len(gts) - len(matched_gt)

    pr = ttp / (ttp + tfp) if (ttp + tfp) > 0 else 0.0
    rc = ttp / (ttp + tfn) if (ttp + tfn) > 0 else 0.0
    f1 = 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0

    # mAP: recall-precision curve
    # Collect all predictions sorted by confidence
    flat_preds = []
    for i, (preds, gts) in enumerate(zip(all_preds, all_gt)):
        for p in preds:
            flat_preds.append(
                {"img": i, "box": p["box"], "score": p["score"], "gts": gts}
            )
    flat_preds.sort(key=lambda x: -x["score"])

    n_gt = sum(len(g) for g in all_gt)
    tp_curve, fp_curve = [], []
    matched_by_img = {i: set() for i in range(len(all_gt))}
    for p in flat_preds:
        img_i = p["img"]
        gts = p["gts"]
        best_iou, best_j = 0.0, -1
        for j, gt in enumerate(gts):
            if j in matched_by_img[img_i]:
                continue
            iou = compute_iou(p["box"], gt)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh:
            tp_curve.append(1)
            fp_curve.append(0)
            matched_by_img[img_i].add(best_j)
        else:
            tp_curve.append(0)
            fp_curve.append(1)

    cum_tp = np.cumsum(tp_curve)
    cum_fp = np.cumsum(fp_curve)
    recalls_c = cum_tp / (n_gt + 1e-9)
    precisions_c = cum_tp / (cum_tp + cum_fp + 1e-9)
    ap = compute_ap(recalls_c.tolist(), precisions_c.tolist())

    _warm = fps_times[5:] if len(fps_times) > 5 else fps_times  # drop warm-up
    fps = len(_warm) / sum(_warm) if _warm else 0.0

    return {
        "mAP@0.5": round(float(ap), 4),
        "mAP@0.5(VOC11)": round(
            float(compute_ap_voc11(recalls_c.tolist(), precisions_c.tolist())), 4
        ),
        "Precision": round(pr, 4),
        "Recall": round(rc, 4),
        "F1": round(f1, 4),
        "TP": ttp,
        "FP": tfp,
        "FN": tfn,
        "FPS": round(fps, 1),
    }


print("Metric helpers ready (COCO-101 AP, synced throughput FPS)")

Metric helpers ready (COCO-101 AP, synced throughput FPS)


In [13]:
# Trains and evaluates YOLOv11m+ECA+CBAM across the three seeds under the same two-phase schedule used for RGDA-YOLOv8m, caching each run's result JSON, and reports mean +/- std. Produces the architecture-dependence result.
import time, gc, pathlib, shutil, json as _json
import numpy as np
from ultralytics import YOLO


def set_seed(seed):
    import random

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _release(model_obj, injector_obj):
    # Fully release a trained model between phases and between seeds.
    if injector_obj is not None:
        try:
            injector_obj.detach()
        except Exception:
            pass
    live_injector["obj"] = None
    if model_obj is not None:
        try:
            model_obj.callbacks = {}  # closures over _attach/_detach
        except Exception:
            pass
        if getattr(model_obj, "trainer", None) is not None:
            model_obj.trainer = None  # dataloaders, EMA, optimiser
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def run_yolov11m_eca_cbam(seed):
    """Runs the full two-phase training + evaluation pipeline for one seed.
    Reuses the same callback-based hook attachment already defined above
    (_attach_on_train_start / _detach_on_train_end) - no changes to that
    logic, just parameterized by seed and wrapped so it can be looped.
    """
    set_seed(seed)
    run_tag = f"YOLOv11m_ECA_CBAM_seed{seed}"

    result_json = os.path.join(OUTPUT_DIR, f"{run_tag}_result.json")
    if os.path.exists(result_json):
        print(f"seed={seed}: loaded from cache -- delete {result_json} for a fresh run")
        with open(result_json) as f:
            return _json.load(f)

    # Phase 1
    for _var in ["model", "injector"]:
        if _var in globals():
            del globals()[_var]
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    model = YOLO("yolo11m.pt")
    model.to(DEVICE)

    live_injector["obj"] = None
    model.add_callback("on_train_start", _attach_on_train_start)
    model.add_callback("on_train_end", _detach_on_train_end)

    print(f"\n{'=' * 60}\nseed={seed}  PHASE 1\n{'=' * 60}")
    t0 = time.time()
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS_FROZEN,
        imgsz=IMG_SIZE,
        batch=BATCH,
        lr0=LR_FROZEN,
        lrf=0.1,
        weight_decay=WEIGHT_DECAY,
        freeze=10,
        device=DEVICE,
        name=f"{run_tag}_phase1",
        exist_ok=True,
        verbose=False,
        plots=False,
        save=True,
        seed=seed,
    )
    model.to(DEVICE)
    injector = live_injector["obj"]
    if injector is not None:
        injector.attach()
    else:
        print("  WARNING: injector is None after Phase 1 -- hooks were not captured!")
    print(f"Phase 1 done in {(time.time() - t0) / 60:.1f} min")

    runs_dir = pathlib.Path("runs/detect")
    p1_best = runs_dir / f"{run_tag}_phase1" / "weights" / "best.pt"
    p1_last = runs_dir / f"{run_tag}_phase1" / "weights" / "last.pt"
    phase1_weights = str(p1_best) if p1_best.exists() else str(p1_last)

    # Phase 2
    _release(model, injector)
    del model, injector
    gc.collect()

    model = YOLO(phase1_weights)
    model.to(DEVICE)
    live_injector["obj"] = None  # FIX: same as Phase 1 -- mutate, don't rebind
    model.add_callback("on_train_start", _attach_on_train_start)
    model.add_callback("on_train_end", _detach_on_train_end)

    print(f"\n{'=' * 60}\nseed={seed}  PHASE 2\n{'=' * 60}")
    t0 = time.time()
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS_FULL,
        imgsz=IMG_SIZE,
        batch=BATCH,
        lr0=LR_FULL,
        lrf=0.01,
        weight_decay=WEIGHT_DECAY,
        freeze=0,
        device=DEVICE,
        name=f"{run_tag}_phase2",
        exist_ok=True,
        verbose=False,
        plots=False,
        save=True,
        seed=seed,
    )
    model.to(DEVICE)
    injector = live_injector["obj"]
    if injector is not None:
        injector.attach()
    else:
        print("  WARNING: injector is None after Phase 2 -- hooks were not captured!")
    print(f"Phase 2 done in {(time.time() - t0) / 60:.1f} min")

    p2_best = runs_dir / f"{run_tag}_phase2" / "weights" / "best.pt"
    p2_last = runs_dir / f"{run_tag}_phase2" / "weights" / "last.pt"
    phase2_weights = str(p2_best) if p2_best.exists() else str(p2_last)

    final_ckpt = os.path.join(SAVE_DIR, f"{run_tag}_final.pt")
    shutil.copy(phase2_weights, final_ckpt)

    if injector is not None:
        att_ckpt = os.path.join(SAVE_DIR, f"{run_tag}_attention_state.pt")
        torch.save(
            {
                "attention_state_dict": injector.attention_modules.state_dict(),
                "cbam_reduction": CBAM_REDUCTION,
                "cbam_kernel": CBAM_KERNEL,
                "yolo_weights_path": phase2_weights,
            },
            att_ckpt,
        )
        print(f"  Attention weights saved: {att_ckpt}")
    else:
        print(
            "  ERROR: no injector -- attention weights NOT saved, evaluation will be on plain model!"
        )

    # Evaluation (uses the already-fixed evaluate_model: mAP from the
    # full curve, Precision/Recall/F1/TP/FP/FN from conf_thresh filtering)
    metrics = evaluate_model(
        model, val_recs, conf_thresh=CONF_DEFAULT, iou_thresh=IOU_THRESH
    )

    with open(result_json, "w") as f:
        _json.dump(metrics, f, indent=2)
    print(f"seed={seed}  mAP@0.5={metrics['mAP@0.5']:.4f}  F1={metrics['F1']:.4f}")

    _release(model, injector)
    del model, injector
    gc.collect()

    return metrics


SEEDS = [42, 123, 456]


_release(globals().get("model"), globals().get("injector"))
for _var in ("model", "injector"):
    globals().pop(_var, None)
gc.collect()

multiseed_results = {}
for seed in SEEDS:
    multiseed_results[seed] = run_yolov11m_eca_cbam(seed)
    # Per-seed results are cached to JSON inside the function, so if the
    # kernel is killed anyway, restart it and re-run this cell: completed
    # seeds are loaded from cache and only the unfinished ones train.

# Aggregate mean +/- std, matching the ExpB multi-seed table format
metrics_by_key = {}
for k in ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall", "F1"]:
    vals = [
        multiseed_results[s].get(k)
        for s in SEEDS
        if multiseed_results[s].get(k) is not None
    ]
    if vals:
        metrics_by_key[k] = (float(np.mean(vals)), float(np.std(vals, ddof=1)))

print("\n" + "=" * 70)
print("  YOLOv11m+ECA+CBAM -- MULTI-SEED RESULTS (n=3, conf=0.25)")
print("=" * 70)
for seed in SEEDS:
    m = multiseed_results[seed]
    print(
        f"  seed={seed}  mAP@0.5={m['mAP@0.5']:.4f}  F1={m['F1']:.4f}  "
        f"P={m['Precision']:.4f}  R={m['Recall']:.4f}"
    )
print("-" * 70)
for k, (mean, std) in metrics_by_key.items():
    print(f"  {k:<15}: {mean:.4f} +/- {std:.4f}")
print("=" * 70)


seed=42  PHASE 1
New https://pypi.org/project/ultralytics/8.4.131 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.24 🚀 Python-3.10.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0

Freezing layer 'model.6.m.0.cv2.conv.weight'
Freezing layer 'model.6.m.0.cv2.bn.weight'
Freezing layer 'model.6.m.0.cv2.bn.bias'
Freezing layer 'model.6.m.0.cv3.conv.weight'
Freezing layer 'model.6.m.0.cv3.bn.weight'
Freezing layer 'model.6.m.0.cv3.bn.bias'
Freezing layer 'model.6.m.0.m.0.cv1.conv.weight'
Freezing layer 'model.6.m.0.m.0.cv1.bn.weight'
Freezing layer 'model.6.m.0.m.0.cv1.bn.bias'
Freezing layer 'model.6.m.0.m.0.cv2.conv.weight'
Freezing layer 'model.6.m.0.m.0.cv2.bn.weight'
Freezing layer 'model.6.m.0.m.0.cv2.bn.bias'
Freezing layer 'model.6.m.0.m.1.cv1.conv.weight'
Freezing layer 'model.6.m.0.m.1.cv1.bn.weight'
Freezing layer 'model.6.m.0.m.1.cv1.bn.bias'
Freezing layer 'model.6.m.0.m.1.cv2.conv.weight'
Freezing layer 'model.6.m.0.m.1.cv2.bn.weight'
Freezing layer 'model.6.m.0.m.1.cv2.bn.bias'
Freezing layer 'model.7.conv.weight'
Freezing layer 'model.7.bn.weight'
Freezing layer 'model.7.bn.bias'
Freezing layer 'model.8.cv1.conv.weight'
Freezing layer 'model.8.cv1.bn.w

/home/vr3/pothole_env/lib/python3.10/site-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/10      4.27G      1.767      2.373      1.722         10        640: 100% ━━━━━━━━━━━━ 47/47 3.6it/s 13.1s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.1it/s 5.3s0.4ss
                   all        186        474     0.0553      0.584      0.044     0.0166

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10      6.78G      1.624      1.958      1.609         38        640: 2% ──────────── 1/47 2.6it/s 0.2s<17.7s

/home/vr3/pothole_env/lib/python3.10/site-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


...

      38/40      10.3G      1.038     0.7381       1.18         16        640: 100% ━━━━━━━━━━━━ 47/47 5.5it/s 8.5s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 9.2it/s 0.6s0.2s
                   all        186        474      0.789      0.677      0.775      0.462

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      39/40      10.3G      1.118      0.836      1.317         28        640: 0% ──────────── 0/47  0.2s

/home/vr3/pothole_env/lib/python3.10/site-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      39/40      10.3G      1.015     0.7214      1.156          5        640: 100% ━━━━━━━━━━━━ 47/47 5.5it/s 8.5s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 9.2it/s 0.7s0.2s
                   all        186        474      0.822      0.694      0.779      0.474

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      40/40      10.3G       1.32     0.8284      1.292         67        640: 0% ──────────── 0/47  0.2s

/home/vr3/pothole_env/lib/python3.10/site-packages/torch/autograd/graph.py:865: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      40/40      10.3G      1.017     0.7095      1.166          9        640: 100% ━━━━━━━━━━━━ 47/47 5.5it/s 8.5s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 9.2it/s 0.7s0.2s
                   all        186        474      0.836      0.686      0.785      0.479

40 epochs completed in 0.106 hours.
Optimizer stripped from /home/vr3/Pothole_Detection/Updated Code 926/Yolov11 Final/runs/detect/YOLOv11m_ECA_CBAM_seed456_phase2/weights/last.pt, 40.5MB
Optimizer stripped from /home/vr3/Pothole_Detection/Updated Code 926/Yolov11 Final/runs/detect/YOLOv11m_ECA_CBAM_seed456_phase2/weights/best.pt, 40.5MB

Validating /home/vr3/Pothole_Detection/Updated Code 926/Yolov11 Final/runs/detect/YOLOv11m_ECA_CBAM_seed456_phase2/weights/best.pt...
Ultralytics 8.4.24 🚀 Python-3.10.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
    